# ChemiAI — Clustering Feature


## Что делаем
Добавляем признак кластера (KMeans) к исходным дескрипторам и обучаем регрессоры на расширенном пространстве признаков.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error


In [ ]:
TARGET_COLS = ["IC50, mM", "CC50, mM", "SI"]
ID_COL = "index"
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
sample_submission = pd.read_csv("data/sample_submission.csv")

feature_cols = [c for c in train.columns if c not in [ID_COL, *TARGET_COLS]]
X_train = train[feature_cols].copy()
y_train = train[TARGET_COLS].copy()
X_test = test[feature_cols].copy()

const_cols = [c for c in X_train.columns if X_train[c].nunique(dropna=False) <= 1]
X_train = X_train.drop(columns=const_cols)
X_test = X_test.drop(columns=const_cols)

imp = SimpleImputer(strategy="median")
Xtr = imp.fit_transform(X_train)
Xte = imp.transform(X_test)
scaler = StandardScaler()
Xtr_s = scaler.fit_transform(Xtr)
Xte_s = scaler.transform(Xte)


## Этап 1: Тюнинг кластеризации и модели

Расширяем поиск числа кластеров `k` и проверяем несколько моделей на augmented-признаках (`исходные фичи + one-hot кластера`).

Цель — найти лучшую конфигурацию по OOF до финального сабмита.

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor


def comp_score(y_true, y_pred):
    return float(np.mean([np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])) for i in range(3)]))


def build_model(model_name):
    if model_name == "hgb":
        return HistGradientBoostingRegressor(max_depth=6, learning_rate=0.03, max_iter=700, random_state=42)
    if model_name == "et":
        return ExtraTreesRegressor(n_estimators=900, random_state=42, n_jobs=-1, min_samples_leaf=2)
    raise ValueError(f"Unknown model: {model_name}")


k_candidates = [4, 6, 8, 10, 12, 16, 20]
model_candidates = ["hgb", "et"]

best_score = np.inf
best_pred = None
best_k = None
best_model = None
best_oof = None

for k in k_candidates:
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    tr_cl = km.fit_predict(Xtr_s)
    te_cl = km.predict(Xte_s)

    tr_oh = np.eye(k)[tr_cl]
    te_oh = np.eye(k)[te_cl]

    Xtr_aug = np.hstack([Xtr, tr_oh])
    Xte_aug = np.hstack([Xte, te_oh])

    for model_name in model_candidates:
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        oof = np.zeros((len(Xtr_aug), 3), dtype=float)
        test_pred = np.zeros((len(Xte_aug), 3), dtype=float)

        for tr_idx, va_idx in kf.split(Xtr_aug):
            X_tr, X_va = Xtr_aug[tr_idx], Xtr_aug[va_idx]
            y_tr = y_train.iloc[tr_idx].values

            for j in range(3):
                yj = np.log1p(y_tr[:, j])
                model = build_model(model_name)
                model.fit(X_tr, yj)

                oof[va_idx, j] = np.expm1(np.clip(model.predict(X_va), 0, 12))
                test_pred[:, j] += np.expm1(np.clip(model.predict(Xte_aug), 0, 12)) / 5

        score = comp_score(y_train.values, oof)
        print(f"k={k}, model={model_name}: OOF score={score:.5f}")

        if score < best_score:
            best_score = score
            best_pred = test_pred
            best_k = k
            best_model = model_name
            best_oof = oof

print(f"Лучший вариант: k={best_k}, model={best_model}, OOF={best_score:.5f}")


## Этап 2: Бленд с лучшим CatBoost-сабмитом

Смешиваем кластерный трек с текущим лучшим `submission_topk.csv` и подбираем вес по OOF. Это помогает взять сильные стороны обоих подходов.

In [ ]:
# 1) Сабмит лучшего чисто кластерного трека
submission_clustering_tuned = sample_submission.copy()
submission_clustering_tuned["IC50"] = best_pred[:, 0]
submission_clustering_tuned["CC50"] = best_pred[:, 1]
submission_clustering_tuned["SI"] = best_pred[:, 2]
submission_clustering_tuned.to_csv("submission_clustering_tuned.csv", index=False)

# 2) Бленд с лучшим CatBoost-треком из файла submission_topk.csv
submission_blend = submission_clustering_tuned.copy()

try:
    sub_topk = pd.read_csv("submission_topk.csv")
    assert list(sub_topk.columns) == ["index", "IC50", "CC50", "SI"]

    # Если OOF второй модели недоступен в этом ноутбуке, берем безопасный вес в пользу нового трека.
    w_cluster = 0.8
    w_topk = 0.2

    for col in ["IC50", "CC50", "SI"]:
        submission_blend[col] = (
            w_cluster * submission_clustering_tuned[col].values
            + w_topk * sub_topk[col].values
        )

    submission_blend.to_csv("submission_clustering_blend_topk.csv", index=False)
    print(f"Сохранены: submission_clustering_tuned.csv и submission_clustering_blend_topk.csv (w_cluster={w_cluster}, w_topk={w_topk})")

except Exception as e:
    print(f"Blend с submission_topk.csv пропущен: {type(e).__name__}: {e}")
    print("Сохранен только submission_clustering_tuned.csv")

submission_clustering_tuned.head()
